# K-Means k=4 sin `cantidad_detecciones`

Este notebook responde al pedido de repetir el clustering multivariado sin usar la columna de focos (`cantidad_detecciones`) como variable de entrada.

La captura del correo se usa solamente como contexto docente. La tarea concreta de este archivo es ajustar K-Means con `k=4` usando variables meteorologicas, y luego mirar `cantidad_detecciones` y el proxy tipo Nesterov como validacion externa de los perfiles.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 40)


## Carga y separacion temporal

Se mantiene la misma separacion temporal del analisis anterior: desarrollo 2018-2023 y prueba externa 2024-2025.


In [ ]:
RUTA_DATOS = "data/processed/firms_open_meteo_weekly/open_meteo_firms_weekly_target_available.parquet"
TRAIN_INICIO = pd.Timestamp("2018-01-01")
TRAIN_FIN = pd.Timestamp("2023-12-31")
POST_INICIO = pd.Timestamp("2024-01-01")
POST_FIN = pd.Timestamp("2025-12-31")

datos = pd.read_parquet(RUTA_DATOS)
datos["fecha_inicio_semana"] = pd.to_datetime(datos["fecha_inicio_semana"], errors="raise")

variables_cluster_ambiental = [
    "temperature_2m_max_mean_weekly",
    "relative_humidity_2m_min_mean_weekly",
    "wind_speed_10m_max_mean_weekly",
    "precipitation_sum_weekly",
]

columnas_requeridas = {"fecha_inicio_semana", "cantidad_detecciones", *variables_cluster_ambiental}
assert columnas_requeridas.issubset(datos.columns)

train = datos.loc[datos["fecha_inicio_semana"].between(TRAIN_INICIO, TRAIN_FIN)].copy()
assert train[list(columnas_requeridas)].isna().sum().sum() == 0

resumen_base = pd.DataFrame({
    "periodo": ["desarrollo"],
    "inicio": [train["fecha_inicio_semana"].min().date()],
    "fin": [train["fecha_inicio_semana"].max().date()],
    "observaciones": [len(train)],
    "departamentos": [train["departamento"].nunique() if "departamento" in train.columns else np.nan],
    "semanas_con_firms": [train["cantidad_detecciones"].gt(0).sum()],
    "porcentaje_firms_positivo": [train["cantidad_detecciones"].gt(0).mean() * 100],
})
resumen_base.round(3)


## Matriz de clustering sin focos

`cantidad_detecciones` no entra en la matriz de K-Means. Se conserva despues solo para describir si los perfiles ambientales tienen distinta distribucion de focos.

La precipitacion se transforma con `log1p` por asimetria; las demas variables entran en su escala original y luego todo se estandariza.


In [ ]:
def preparar_variables_cluster(dataframe):
    matriz = dataframe[variables_cluster_ambiental].astype(float).copy()
    matriz["precipitation_sum_weekly"] = np.log1p(matriz["precipitation_sum_weekly"])
    return matriz

X_sin_escalar = preparar_variables_cluster(train)
scaler_ambiental = StandardScaler()
X_ambiental = scaler_ambiental.fit_transform(X_sin_escalar)

pd.DataFrame(X_ambiental, columns=variables_cluster_ambiental).agg(["mean", "std"]).round(3)


## Evaluacion de `k`

Se muestran metricas para `k=2` a `k=6`. En esta variante puramente ambiental, `k=3` suele quedar levemente mejor en Silhouette, pero el pedido docente es repetir el modelo con `k=4`; por eso se fija `k=4` para poder comparar cuatro perfiles.


In [ ]:
resultados_k = []
modelos_k = {}

for k in range(2, 7):
    modelo = KMeans(n_clusters=k, random_state=42, n_init=20)
    etiquetas = modelo.fit_predict(X_ambiental)
    modelos_k[k] = modelo
    conteos = pd.Series(etiquetas).value_counts().sort_index()
    resultados_k.append({
        "k": k,
        "inercia": modelo.inertia_,
        "silhouette": silhouette_score(X_ambiental, etiquetas),
        "davies_bouldin": davies_bouldin_score(X_ambiental, etiquetas),
        "minimo_cluster": int(conteos.min()),
        "maximo_cluster": int(conteos.max()),
        "tamanos": conteos.tolist(),
        "porcentajes": conteos.div(len(etiquetas)).mul(100).round(2).tolist(),
    })

resultados_k = pd.DataFrame(resultados_k)
resultados_k.round(4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(resultados_k["k"], resultados_k["inercia"], marker="o", color="#4c78a8")
axes[0].set_title("Metodo del codo - K-Means ambiental")
axes[0].set_xlabel("Numero de clusters (k)")
axes[0].set_ylabel("Inercia")
axes[0].set_xticks(resultados_k["k"])
axes[0].grid(alpha=0.3)

axes[1].plot(resultados_k["k"], resultados_k["silhouette"], marker="o", color="#e6550d")
axes[1].axvline(4, color="#555555", linestyle="--", linewidth=1, label="k=4 pedido")
axes[1].set_title("Silhouette - K-Means ambiental")
axes[1].set_xlabel("Numero de clusters (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_xticks(resultados_k["k"])
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Ajuste final con `k=4`

Las letras se asignan por interpretacion meteorologica, no por nivel creciente de detecciones: frio/lluvioso, frio/seco, calido/lluvioso y calido/seco.


In [ ]:
K_AMBIENTAL = 4
kmeans_ambiental = modelos_k[K_AMBIENTAL]
train_ambiental = train.copy()
train_ambiental["cluster_kmeans_sin_firms"] = kmeans_ambiental.labels_

resumen_raw = train_ambiental.groupby("cluster_kmeans_sin_firms").agg(
    observaciones=("cantidad_detecciones", "count"),
    temp_media=("temperature_2m_max_mean_weekly", "mean"),
    humedad_media=("relative_humidity_2m_min_mean_weekly", "mean"),
    viento_media=("wind_speed_10m_max_mean_weekly", "mean"),
    precipitacion_media=("precipitation_sum_weekly", "mean"),
)

clusters_frio = resumen_raw.nsmallest(2, "temp_media").index.tolist()
cluster_frio_lluvioso = resumen_raw.loc[clusters_frio, "precipitacion_media"].idxmax()
cluster_frio_seco = [c for c in clusters_frio if c != cluster_frio_lluvioso][0]
clusters_calido = resumen_raw.nlargest(2, "temp_media").index.tolist()
cluster_calido_lluvioso = resumen_raw.loc[clusters_calido, "precipitacion_media"].idxmax()
cluster_calido_seco = [c for c in clusters_calido if c != cluster_calido_lluvioso][0]

mapa_perfiles = {
    int(cluster_frio_lluvioso): "Perfil A",
    int(cluster_frio_seco): "Perfil B",
    int(cluster_calido_lluvioso): "Perfil C",
    int(cluster_calido_seco): "Perfil D",
}
descripcion_perfiles = {
    "Perfil A": "frio-humedo/lluvioso y ventoso",
    "Perfil B": "frio-seco con baja precipitacion",
    "Perfil C": "calido-humedo/lluvioso",
    "Perfil D": "calido-seco, mayor deficit atmosferico",
}
orden_perfiles = ["Perfil A", "Perfil B", "Perfil C", "Perfil D"]

train_ambiental["perfil_kmeans_sin_firms"] = train_ambiental["cluster_kmeans_sin_firms"].map(mapa_perfiles)

mapeo_df = pd.DataFrame({
    "cluster_original": list(mapa_perfiles.keys()),
    "perfil": list(mapa_perfiles.values()),
})
mapeo_df["descripcion"] = mapeo_df["perfil"].map(descripcion_perfiles)
mapeo_df.set_index("perfil").reindex(orden_perfiles).reset_index()


In [ ]:
perfil_ambiental = train_ambiental.groupby("perfil_kmeans_sin_firms").agg(
    observaciones=("cantidad_detecciones", "count"),
    temperatura_media=("temperature_2m_max_mean_weekly", "mean"),
    humedad_minima_media=("relative_humidity_2m_min_mean_weekly", "mean"),
    viento_maximo_medio=("wind_speed_10m_max_mean_weekly", "mean"),
    precipitacion_media=("precipitation_sum_weekly", "mean"),
).reindex(orden_perfiles)
perfil_ambiental["porcentaje_dataset"] = perfil_ambiental["observaciones"].div(len(train_ambiental)).mul(100)
perfil_ambiental = perfil_ambiental[
    ["observaciones", "porcentaje_dataset", "temperatura_media", "humedad_minima_media", "viento_maximo_medio", "precipitacion_media"]
]
perfil_ambiental.round(3)


In [ ]:
centroides_ambientales = pd.DataFrame(
    scaler_ambiental.inverse_transform(kmeans_ambiental.cluster_centers_),
    columns=variables_cluster_ambiental,
)
centroides_ambientales["precipitation_sum_weekly"] = np.expm1(centroides_ambientales["precipitation_sum_weekly"])
centroides_ambientales["perfil"] = centroides_ambientales.index.map(mapa_perfiles)
centroides_ambientales = centroides_ambientales.set_index("perfil").reindex(orden_perfiles)
centroides_ambientales.round(3)


In [ ]:
centroides_std = pd.DataFrame(kmeans_ambiental.cluster_centers_, columns=variables_cluster_ambiental)
centroides_std["perfil"] = centroides_std.index.map(mapa_perfiles)
centroides_std = centroides_std.set_index("perfil").reindex(orden_perfiles)

fig, ax = plt.subplots(figsize=(9, 3.8))
imagen = ax.imshow(centroides_std, cmap="RdBu_r", aspect="auto", vmin=-1.5, vmax=1.5)
ax.set_xticks(range(len(centroides_std.columns)), labels=["Temperatura", "Humedad", "Viento", "Precipitacion"], rotation=25, ha="right")
ax.set_yticks(range(len(orden_perfiles)), labels=orden_perfiles)
ax.set_title("Centroides estandarizados - K-Means sin cantidad_detecciones")
for fila in range(centroides_std.shape[0]):
    for columna in range(centroides_std.shape[1]):
        ax.text(columna, fila, f"{centroides_std.iloc[fila, columna]:.2f}", ha="center", va="center", fontsize=9)
fig.colorbar(imagen, ax=ax, label="Media estandarizada")
plt.tight_layout()
plt.show()


## Focos como validacion externa

A partir de aca se vuelve a mirar `cantidad_detecciones`, pero no como variable de entrada. Esto responde directamente a la pregunta docente: ver si los perfiles meteorologicos tienen distribuciones distintas de actividad FIRMS.


In [ ]:
total_detecciones = train_ambiental["cantidad_detecciones"].sum()
actividad = train_ambiental.groupby("perfil_kmeans_sin_firms")["cantidad_detecciones"].agg(
    observaciones="count",
    semanas_positivas=lambda x: x.gt(0).sum(),
    semanas_cero=lambda x: x.eq(0).sum(),
    media="mean",
    mediana="median",
    p75=lambda x: x.quantile(0.75),
    p90=lambda x: x.quantile(0.90),
    maximo="max",
    detecciones_totales="sum",
).reindex(orden_perfiles)
actividad["porcentaje_dataset"] = actividad["observaciones"].div(len(train_ambiental)).mul(100)
actividad["porcentaje_semanas_positivas"] = actividad["semanas_positivas"].div(actividad["observaciones"]).mul(100)
actividad["porcentaje_semanas_cero"] = actividad["semanas_cero"].div(actividad["observaciones"]).mul(100)
actividad["porcentaje_total_detecciones"] = actividad["detecciones_totales"].div(total_detecciones).mul(100)
actividad.round(3)


In [ ]:
positivos = train_ambiental.loc[train_ambiental["cantidad_detecciones"].gt(0)].groupby("perfil_kmeans_sin_firms")["cantidad_detecciones"].agg(
    semanas_positivas="count",
    media_positivas="mean",
    mediana_positivas="median",
    p75_positivas=lambda x: x.quantile(0.75),
    p90_positivas=lambda x: x.quantile(0.90),
    maximo_positivas="max",
).reindex(orden_perfiles)
positivos.round(3)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4))
actividad["porcentaje_semanas_positivas"].plot(kind="bar", ax=axes[0], color="#4c78a8")
axes[0].set_title("Semanas con actividad FIRMS por perfil")
axes[0].set_xlabel("Perfil ambiental")
axes[0].set_ylabel("FIRMS > 0 (%)")
axes[0].tick_params(axis="x", rotation=0)
axes[0].grid(axis="y", alpha=0.25)

valores_box = [
    train_ambiental.loc[
        train_ambiental["perfil_kmeans_sin_firms"].eq(perfil) & train_ambiental["cantidad_detecciones"].gt(0),
        "cantidad_detecciones",
    ]
    for perfil in orden_perfiles
]
axes[1].boxplot(valores_box, tick_labels=orden_perfiles, showfliers=True)
axes[1].set_yscale("log")
axes[1].set_title("Intensidad FIRMS en semanas positivas")
axes[1].set_xlabel("Perfil ambiental")
axes[1].set_ylabel("Detecciones por semana positiva (escala log)")
axes[1].grid(axis="y", alpha=0.25, which="both")
plt.tight_layout()
plt.show()


## Proxy inspirado en Nesterov

Este proxy se usa como referencia interpretativa y no como indice oficial calibrado. Permite revisar si los perfiles secos/calidos quedan asociados a mayor potencial meteorologico de incendio.


In [ ]:
T = train_ambiental["temperature_2m_max_mean_weekly"]
H = train_ambiental["relative_humidity_2m_min_mean_weekly"]
E = 6.112 * np.exp((17.67 * T) / (T + 243.5))
d = E * (1 - H / 100)
train_ambiental["nesterov_proxy"] = 7 * d * T

nesterov = train_ambiental.groupby("perfil_kmeans_sin_firms")["nesterov_proxy"].agg(
    observaciones="count",
    media="mean",
    mediana="median",
    p25=lambda x: x.quantile(0.25),
    p75=lambda x: x.quantile(0.75),
).reindex(orden_perfiles)
nesterov.round(3)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4))
valores_nesterov = [train_ambiental.loc[train_ambiental["perfil_kmeans_sin_firms"].eq(perfil), "nesterov_proxy"] for perfil in orden_perfiles]
axes[0].boxplot(valores_nesterov, tick_labels=orden_perfiles, showfliers=False)
axes[0].set_title("Proxy Nesterov por perfil")
axes[0].set_xlabel("Perfil ambiental")
axes[0].set_ylabel("nesterov_proxy")

nesterov[["media", "mediana"]].plot(kind="bar", ax=axes[1], color=["#e6550d", "#3182bd"])
axes[1].set_title("Media y mediana del proxy")
axes[1].set_xlabel("Perfil ambiental")
axes[1].set_ylabel("nesterov_proxy")
axes[1].tick_params(axis="x", rotation=0)
axes[1].grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## Estacionalidad y prueba externa 2024-2025

La distribucion mensual sirve para verificar que los perfiles se comportan como estaciones/regimenes meteorologicos. Luego se aplica el modelo congelado al periodo 2024-2025.


In [ ]:
train_ambiental["mes"] = train_ambiental["fecha_inicio_semana"].dt.month
meses = pd.crosstab(
    train_ambiental["mes"],
    train_ambiental["perfil_kmeans_sin_firms"],
    normalize="index",
).reindex(columns=orden_perfiles, fill_value=0).mul(100)
meses.round(1)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
meses.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
ax.set_title("Distribucion mensual de perfiles ambientales - desarrollo 2018-2023")
ax.set_xlabel("Mes")
ax.set_ylabel("Porcentaje de observaciones")
ax.legend(title="Perfil", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
post = datos.loc[datos["fecha_inicio_semana"].between(POST_INICIO, POST_FIN)].copy()
X_post = preparar_variables_cluster(post)
X_post_scaled = scaler_ambiental.transform(X_post)
post["cluster_kmeans_sin_firms"] = kmeans_ambiental.predict(X_post_scaled)
post["perfil_kmeans_sin_firms"] = post["cluster_kmeans_sin_firms"].map(mapa_perfiles)

distribucion_temporal = pd.DataFrame({
    "% desarrollo 2018-2023": train_ambiental["perfil_kmeans_sin_firms"].value_counts(normalize=True).reindex(orden_perfiles).mul(100),
    "% prueba 2024-2025": post["perfil_kmeans_sin_firms"].value_counts(normalize=True).reindex(orden_perfiles).mul(100),
})
distribucion_temporal["diferencia pp"] = distribucion_temporal["% prueba 2024-2025"] - distribucion_temporal["% desarrollo 2018-2023"]
distribucion_temporal.round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
distribucion_temporal[["% desarrollo 2018-2023", "% prueba 2024-2025"]].plot(kind="bar", ax=ax, color=["#4c78a8", "#f58518"])
ax.set_title("Aplicacion externa del modelo congelado")
ax.set_xlabel("Perfil ambiental")
ax.set_ylabel("Porcentaje de observaciones")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
post_perfiles = post.groupby("perfil_kmeans_sin_firms").agg(
    observaciones=("cantidad_detecciones", "count"),
    porcentaje_firms_positivo=("cantidad_detecciones", lambda x: x.gt(0).mean() * 100),
    media_firms=("cantidad_detecciones", "mean"),
    detecciones_totales=("cantidad_detecciones", "sum"),
    temperatura_media=("temperature_2m_max_mean_weekly", "mean"),
    humedad_minima_media=("relative_humidity_2m_min_mean_weekly", "mean"),
    precipitacion_media=("precipitation_sum_weekly", "mean"),
).reindex(orden_perfiles)
post_perfiles.round(3)


In [ ]:
pca = PCA(n_components=2)
componentes = pca.fit_transform(X_ambiental)
pca_df = pd.DataFrame(componentes, columns=["PC1", "PC2"])
pca_df["perfil"] = train_ambiental["perfil_kmeans_sin_firms"].to_numpy()

fig, ax = plt.subplots(figsize=(7.5, 5.5))
for perfil in orden_perfiles:
    parte = pca_df[pca_df["perfil"].eq(perfil)]
    ax.scatter(parte["PC1"], parte["PC2"], s=12, alpha=0.35, label=perfil)
ax.set_title("PCA de perfiles ambientales")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% varianza)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% varianza)")
ax.legend(title="Perfil")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## Lectura esperada con los datos actuales

Con los datos actuales del repositorio, la corrida sin `cantidad_detecciones` produce perfiles meteorologicos, no niveles crecientes de focos:

- `Perfil A`: frio, humedo/lluvioso y ventoso. Aproximadamente 24.9% de las observaciones y 15.6% de las detecciones acumuladas.
- `Perfil B`: frio y seco. Aproximadamente 28.9% de las observaciones y 37.1% de las detecciones acumuladas.
- `Perfil C`: calido y lluvioso. Aproximadamente 26.7% de las observaciones y 23.4% de las detecciones acumuladas.
- `Perfil D`: calido y seco. Aproximadamente 19.5% de las observaciones, 24.0% de las detecciones acumuladas y el mayor proxy Nesterov medio.

La mediana de `cantidad_detecciones` permanece en 0 para los cuatro perfiles. Esto refuerza la lectura sugerida por la profesora: al excluir los focos del ajuste, K-Means ya no fuerza un perfil "con focos" frente a otros "sin focos"; separa regimenes meteorologicos y despues la actividad FIRMS se evalua como dimension adicional.
